# Einselection — the classical basis is chosen by the interaction

**The punchline.** Ask which quantum states survive contact with the environment and the
answer is *not a property of the states*. It is a property of the **coupling**. Change
what the environment asks about, and a completely different set of states becomes the
robust, classical-looking one — while the previously classical states become fragile
superpositions.

Nothing makes $\lvert 0\rangle$ more real, more definite, or more classical than
$\lvert +\rangle$. The world is full of definite positions rather than superpositions of
positions because the interactions that dominate — photons and air molecules bouncing off
things — couple through position. Position is not privileged by the laws of physics. It
is privileged by what does the bouncing.

Background: **[06 — Why the world looks classical](../06-decoherence.ipynb)** §8, and
**[decoherence_dial](decoherence_dial.ipynb)** for the coupling this is built on.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from qsim import Circuit
from qsim.decoherence import pointer_coupling
from qsim.gates import H, S, X

np.set_printoptions(precision=3, suppress=True)

## 1. One coupling with a knob on it

`pointer_coupling(q, env, theta=t, basis=b)` is the dephasing coupling of
[decoherence_dial](decoherence_dial.ipynb) **conjugated into another basis**: rotate so
that the basis you care about becomes the computational one, dephase there, rotate back.
In `qsim` that sandwich is spelled with `qsim.within`, and it is three lines
(see [04 — Programs made of gates](../04-combinators.ipynb) §6):

```python
with qsim.within(_onto_computational_basis, q, basis=basis):
    dephasing_coupling(q, env, theta=theta)
```

With `basis="z"` the wrapper is empty and this is plain dephasing — the environment asks
"are you $\lvert 0\rangle$ or $\lvert 1\rangle$?". With `basis="x"` the wrapper is $H$,
and it asks "are you $\lvert +\rangle$ or $\lvert -\rangle$?". With `basis="y"` it asks
about the third axis.

The environment qubit does not care what the answer means. It records the answer to
whichever question the physics of the interaction poses.

## 2. Six states, three questions

We prepare each of the six axis states of the Bloch sphere and expose it to a *perfect*
record ($\theta = \pi$) in each of the three bases. The readout is the **system entropy**:
0 bits means the qubit came through untouched, with a state of its own; 1 bit means it is
now maximally entangled with the environment and has no state of its own at all.

In [ ]:
# Each preparation is a list of gates applied to a fresh |0>. The six states are the
# ends of the three Bloch axes: |0>,|1> along z, |+>,|-> along x, |+i>,|-i> along y.
preparations = [
    ("|0>", []),
    ("|1>", [X]),
    ("|+>", [H]),
    ("|->", [X, H]),
    ("|+i>", [H, S]),
    ("|-i>", [X, H, S]),
]
bases = ["z", "x", "y"]


def survive(prepare: list, basis: str, theta: float = np.pi) -> float:
    """System entropy in bits after a `basis` coupling of strength `theta`."""
    qc = Circuit(name="einselection", seed=11)
    q = qc.alloc("q")
    env = qc.environment(1)
    for op in prepare:
        op(q)
    pointer_coupling(q, env[0], theta=theta, basis=basis)
    return qc.inspect.system_entropy()


# A row per prepared state, a column per coupling basis.
table = np.array([[survive(prepare, basis) for basis in bases]
                  for _, prepare in preparations])

print("system entropy in bits after a perfect record (0 = survived, 1 = destroyed)")
print(f"{'':>6}" + "".join(f"{'basis ' + b:>12}" for b in bases))
for (label, _), row in zip(preparations, table, strict=True):
    print(f"{label:>6}" + "".join(f"{value:12.3f}" for value in row))

Eighteen numbers, and every one of them is either 0 or 1 — nothing in between. The same
table as a picture makes the pattern impossible to miss.

In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 4.4))
image = ax.imshow(table, cmap="magma", vmin=0.0, vmax=1.0)
for i in range(table.shape[0]):
    for j in range(table.shape[1]):
        ax.text(j, i, f"{table[i, j]:.2f}", ha="center", va="center",
                color="white" if table[i, j] < 0.6 else "black")
ax.set_xticks(range(len(bases)), [f"couple through {b}" for b in bases])
ax.set_yticks(range(len(preparations)), [label for label, _ in preparations])
ax.set_title("who survives a perfect record?\n(0 bits = untouched, 1 bit = destroyed)")
fig.colorbar(image, ax=ax, label="system entropy (bits)", fraction=0.046, pad=0.04)
fig.tight_layout()

Read it as a permutation matrix and the point is unmissable. **Each coupling leaves
exactly two states alone and destroys the other four.** Which two depends only on the
column — on the interaction — and not at all on anything intrinsic to the rows.

The two survivors of a coupling are its **pointer states**: the states the environment
copies rather than disturbs, and therefore the only ones that can persist long enough to
be what we call "definite". Every other state gets turned into a mixture, which is to say
into something that *looks* like classical ignorance about which pointer state it "really"
is.

The off-axis entries are worth a second look too: $\lvert 0\rangle$ under an $x$ coupling
gets a *full* bit of entropy, exactly as much as $\lvert +\rangle$ does under a $z$
coupling. The destruction is symmetric. There is no sense in which the computational
basis is the true one and the others are exotic.

## 3. The dial, in two bases at once

The table used a perfect record. Turn the strength down and the two columns become mirror
images of each other.

In [ ]:
thetas = np.linspace(0.0, np.pi, 61)
fig, axes = plt.subplots(1, 2, figsize=(11.0, 3.6), sharey=True)

for ax, basis in zip(axes, ["z", "x"], strict=True):
    for label, prepare in preparations[:4]:
        curve = [survive(prepare, basis, theta) for theta in thetas]
        ax.plot(thetas, curve, lw=2, label=label)
    ax.set_title(f'coupling through {basis}: pointer states are '
                 + ("|0>, |1>" if basis == "z" else "|+>, |->"))
    ax.set_xlabel(r"$\theta$ — how good a record the environment makes")
    ax.set_xticks([0, np.pi / 2, np.pi], ["0", "π/2", "π"])
    ax.legend(fontsize=8, loc="upper left")

axes[0].set_ylabel("system entropy (bits)")
axes[0].set_ylim(-0.05, 1.25)
fig.tight_layout()

In the left panel $\lvert 0\rangle$ and $\lvert 1\rangle$ lie flat along the bottom for
every strength of coupling — the environment can look as hard as it likes and they do not
change — while $\lvert +\rangle$ and $\lvert -\rangle$ climb to a full bit. In the right
panel the two pairs swap roles exactly.

A pointer state is not merely *less damaged*. It is untouched at every $\theta$, which is
the strong form of the claim: the interaction has an eigenbasis, and states in it are
copied into the environment rather than disturbed by it. That is why they can be
*redundantly* recorded — many independent fragments of the environment can all carry the
same answer — and redundant recording is what makes a fact feel objective. (Zurek calls
that quantum Darwinism; it is the natural next demo once the library can couple one qubit
to many environments.)

## 4. Why this is the beginning of an answer to "why does the world look classical?"

Put the three claims side by side:

1. A superposition survives only if nothing in the world records which branch it is in
   ([decoherence_dial](decoherence_dial.ipynb)).
2. Which superpositions get recorded is fixed by *how* the environment couples — this
   notebook.
3. The couplings that dominate everyday physics are scattering processes: a photon
   bounces off an object and carries away information about **where it was**.

So the states that survive in the world we live in are the position-definite ones. A cat
in a superposition of alive and dead is not forbidden by any law; it is decohered in
something like $10^{-20}$ seconds by the air in the room, because air molecules scatter
off a cat's position. Change the dominant coupling and the classical variables change
with it — for a superconducting qubit in a well-isolated cavity, the pointer states are
not positions at all.

**Two honest caveats.** First, this explains why we see *these* alternatives rather than
those, not why we see *one* of them. The global state after decoherence is still a
superposition of all the branches; einselection picks the basis, not the outcome. That
gap is the measurement problem, and it is exactly where
[wigners_friend](wigners_friend.ipynb) picks up. Second, real pointer states are only
approximately stable — the "predictability sieve" ranks them by how slowly they degrade,
and a two-qubit model makes them look cleaner than they are.

## Where to go next

- **[quantum_eraser](quantum_eraser.ipynb)**: the records made here are all undoable,
  which is the other half of the story.
- **[wigners_friend](wigners_friend.ipynb)**: what einselection does *not* explain.
- **[06 — Why the world looks classical](../06-decoherence.ipynb)** §10: the other noise
  channels, and how they erode the Bloch vector differently.

## Assertions

The claims above, re-checked numerically.

In [ ]:
# 1. Each coupling leaves exactly two states untouched and destroys the other four.
for column, basis in enumerate(bases):
    survivors = np.isclose(table[:, column], 0.0, atol=1e-12)
    assert survivors.sum() == 2, f"basis {basis} should have exactly 2 pointer states"
    assert np.allclose(table[~survivors, column], 1.0), f"basis {basis}: partial damage"

# 2. The survivors are the axis pair matching the coupling, in order z, x, y.
expected = {"z": {"|0>", "|1>"}, "x": {"|+>", "|->"}, "y": {"|+i>", "|-i>"}}
labels = np.array([label for label, _ in preparations])
for column, basis in enumerate(bases):
    assert set(labels[np.isclose(table[:, column], 0.0, atol=1e-12)]) == expected[basis]

# 3. A pointer state is untouched at *every* coupling strength, not merely at the end.
for theta in np.linspace(0.0, np.pi, 9):
    assert np.isclose(survive([], "z", theta), 0.0, atol=1e-12)     # |0> under z
    assert np.isclose(survive([H], "x", theta), 0.0, atol=1e-12)    # |+> under x

# 4. Off-basis states decohere by exactly the same law as in decoherence_dial.
for theta in np.linspace(0.0, np.pi, 9):
    qc = Circuit(seed=11)
    q = qc.alloc()
    env = qc.environment(1)
    pointer_coupling(q, env[0], theta=theta, basis="x")   # |0> is a superposition here
    # Coherence in the x basis is the |+>/|-> off-diagonal; on the Bloch sphere that is
    # the z component, which should follow cos(theta/2) just as x did before.
    assert np.isclose(qc.inspect.bloch_vector(q)[2], np.cos(theta / 2), atol=1e-12)

print("all assertions passed")